## Enhanced Visualization Features

The DNA browser now includes **missed scoring visualization** with the following new features:

### 🔍 Missed Scoring Analysis
- **Orange highlighting**: Time periods where scoring criteria were missed
- **Detailed annotations**: Show expected vs actual neuron behavior (e.g., "Miss: W1 G0")
- **Total missed points**: Displayed in plot titles
- **🎯 Gold borders**: Highlight neurons used for fitness evaluation

### 🎛️ Enhanced Controls
- **Condition selector**: View experimental, control, or both conditions
- **Missed scoring**: Automatically enabled for voltage trace visualization
- **Interactive diagnostics**: See exactly where and why points were lost

### 📊 Visual Indicators
- **Criteria neurons**: Marked with 🎯 symbol in subplot titles
- **Missed point counts**: Shown next to each neuron name
- **Orange annotations**: Detailed "Wanted vs Got" information for each missed period

# Multiple GA Results Analysis & DNA Pruning

This notebook:
1. Searches through GA result folders for all `aggregated_results.pkl` files
2. Finds all DNA vectors exceeding a score threshold
3. Runs weight pruning on each high-scoring DNA
4. Creates interactive plots with slider to browse between different DNA vectors

In [10]:
# Import required libraries
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as pyo
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox, Output, Button
from IPython.display import display, clear_output
import time
from copy import deepcopy

# Import project modules
from src.constants import *
from src.neuron import *
from src.network import *
from src.validation import *
from src.genetic_algorithm import *

# Import weight pruning functionality
import sys
sys.path.append('.')
from weight_pruning import WeightPruner, evaluate_single_dna, prune_dna_vectors, evaluate_single_dna_fast

# Import analysis modules
from dna_analyzer import find_all_aggregated_results, extract_high_scoring_dnas
from dna_simulation import run_dna_with_voltage_tracking, generate_all_simulation_results
from dna_visualization import create_voltage_plot, create_directed_graph_from_dna, create_network_plot
from dna_browser import create_dual_dna_browser

print("✅ All imports successful")

✅ All imports successful


## Configuration

In [11]:
# Configuration
RESULTS_FOLDER = "results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018"  # Change this to your results folder
SCORE_THRESHOLD = 970  # Minimum score threshold for DNA selection
PRUNING_THRESHOLD = 975  # Minimum score to maintain during pruning
MAX_DNAS_TO_PROCESS = 3  # Limit number of DNAs to prevent overwhelming

# Duplicate removal options (Step 1)
REMOVE_EXACT_DUPLICATES = True  # Remove DNAs with identical vectors
UNIQUE_CONFIGURATIONS_ONLY = True  # Keep only best DNA for each unique non-zero pattern

# Post-pruning filtering options (Step 2)
POST_PRUNING_UNIQUE_CONFIGS = True  # Apply unique configuration filtering after pruning
MAX_PRUNED_WEIGHTS = 18  # Maximum non-zero weights allowed in pruned DNA (None = no limit)

print(f"Configuration:")
print(f"  Results folder: {RESULTS_FOLDER}")
print(f"  Score threshold: {SCORE_THRESHOLD}")
print(f"  Pruning threshold: {PRUNING_THRESHOLD}")
print(f"  Max DNAs to process: {MAX_DNAS_TO_PROCESS}")
print(f"  Remove exact duplicates: {REMOVE_EXACT_DUPLICATES}")
print(f"  Unique configurations only: {UNIQUE_CONFIGURATIONS_ONLY}")
print(f"  Post-pruning unique configs: {POST_PRUNING_UNIQUE_CONFIGS}")
print(f"  Max pruned weights: {MAX_PRUNED_WEIGHTS or 'No limit'}")

Configuration:
  Results folder: results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018
  Score threshold: 970
  Pruning threshold: 975
  Max DNAs to process: 3
  Remove exact duplicates: True
  Unique configurations only: True
  Post-pruning unique configs: True
  Max pruned weights: 18


## Step 1: Find All High-Scoring DNA Vectors

In [12]:
# Execute step 1 using functions from dna_analyzer module
print("🔍 Step 1: Finding all high-scoring DNA vectors (OPTIMIZED)...")
aggregated_files = find_all_aggregated_results(RESULTS_FOLDER)
high_scoring_dnas = extract_high_scoring_dnas(
    aggregated_files, 
    SCORE_THRESHOLD, 
    remove_duplicates=REMOVE_EXACT_DUPLICATES,
    unique_configs_only=UNIQUE_CONFIGURATIONS_ONLY
)

# Limit number of DNAs to process
if len(high_scoring_dnas) > MAX_DNAS_TO_PROCESS:
    print(f"\n⚠️  Found {len(high_scoring_dnas)} DNAs, limiting to top {MAX_DNAS_TO_PROCESS} for performance")
    high_scoring_dnas = high_scoring_dnas[:MAX_DNAS_TO_PROCESS]

print(f"\n✅ Step 1 complete: {len(high_scoring_dnas)} DNA vectors ready for pruning")

🔍 Step 1: Finding all high-scoring DNA vectors (OPTIMIZED)...
📁 Found 1 aggregated_results.pkl files
  results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018/aggregated_results.pkl
🚀 Loading 1 files in parallel...

🎯 Found 43341 DNA vectors with score >= 970
🔍 Removing exact duplicates from 43341 DNAs...
  ✅ Removed 541 exact duplicates, 42800 unique DNAs remain
🎯 Filtering for unique configurations from 42800 DNAs...
  Found 25 unique weight configurations:
    Config 1: 51 non-zero weights, 541 DNAs (scores: 970-973), kept best: 973
    Config 2: 50 non-zero weights, 876 DNAs (scores: 970-972), kept best: 972
    Config 3: 51 non-zero weights, 247 DNAs (scores: 970-972), kept best: 972
    Config 4: 50 non-zero weights, 1670 DNAs (scores: 970-974), kept best: 974
    Config 5: 49 non-zero weights, 178 DNAs (scores: 970-974), kept best: 974
    Config 6: 51 non-zero weights, 1 DNAs (scores: 970-970), kept best: 970
    Config 7: 50 non-zero weights, 25 DNAs (score

## Step 2: Prune Each High-Scoring DNA

In [13]:
# DNA pruning functions are now imported from weight_pruning module
# See weight_pruning.py for the GenerationBasedPruner class implementation
pruned_results, successful_vectors = prune_dna_vectors(high_scoring_dnas, PRUNING_THRESHOLD, 
                                                      method="fast_greedy", score_tolerance=10)


⚡ Starting FAST GREEDY pruning for 3 DNA vectors...
🎯 Strategy: Phase 1 (equal-or-better) + Phase 2 (tolerance: -10)
🎯 Success threshold: 975

⚡ Fast greedy pruning DNA 1 (Score: 978, Tolerance: -10)...
  🎯 Starting with 48 weights, score: 978

  🔶 PHASE 1: Maintaining equal-or-better score...
    🔄 Phase 1 Pass 1: Testing 48 weights...
      ✅ Removed weight 10 (value:  -2) -> Score: 978, Non-zero: 47
      ✅ Removed weight 22 (value:   3) -> Score: 978, Non-zero: 46
      ✅ Removed weight 36 (value:   6) -> Score: 978, Non-zero: 45
      ✅ Removed weight 43 (value:  -7) -> Score: 978, Non-zero: 44
      ✅ Removed weight 39 (value:  -9) -> Score: 978, Non-zero: 43
      ✅ Removed weight 31 (value:  11) -> Score: 978, Non-zero: 42
      ✅ Removed weight 48 (value:  15) -> Score: 978, Non-zero: 41
      ✅ Removed weight 3 (value:  25) -> Score: 978, Non-zero: 40
      ✅ Removed weight 42 (value: -25) -> Score: 978, Non-zero: 39
      ✅ Removed weight 28 (value: -55) -> Score: 978, Non-z

In [14]:
# Use FAST GREEDY pruning method with score tolerance for better weight reduction
# This method removes smallest weights first, with two phases:
# Phase 1: Maintain equal-or-better scores  
# Phase 2: Allow score decrease up to tolerance for more aggressive pruning

# Create target DNAs from successful vectors that meet weight criteria
target_dnas = []

# First, check successful vectors found DURING pruning (these already meet score threshold)
for sv in successful_vectors:
    if sv['nonzero_weights'] <= MAX_PRUNED_WEIGHTS:
        # Convert successful vector to same format as pruned_results for compatibility
        target_dna = {
            'original_dna': sv,  # Using the successful vector info as original
            'pruned_dna': sv['dna'],
            'original_score': sv['score'],  # For successful vectors, "original" is their score
            'pruned_score': sv['score'],
            'original_nonzero': sv['nonzero_weights'],
            'pruned_nonzero': sv['nonzero_weights'],
            'weights_removed': 0,  # No additional removal needed
            'final_exp_score': sv['exp_score'],
            'final_cont_score': sv['cont_score'],
            'id': sv['original_dna_id']
        }
        target_dnas.append(target_dna)

# Also check final pruned results that meet both criteria
for result in pruned_results:
    meets_score = result['pruned_score'] >= PRUNING_THRESHOLD
    meets_weight = result['pruned_nonzero'] <= MAX_PRUNED_WEIGHTS
    
    if meets_score and meets_weight:
        # Check if not already in target_dnas (avoid duplicates)
        existing_ids = [td['id'] for td in target_dnas]
        if result['id'] not in existing_ids:
            target_dnas.append(result)

print(f"\n🎯 TARGET DNAs FOUND: {len(target_dnas)} meet both criteria:")
print(f"  Score >= {PRUNING_THRESHOLD}: ✅")
print(f"  Weights <= {MAX_PRUNED_WEIGHTS}: ✅")
print(f"  From successful vectors: {len([td for td in target_dnas if td['weights_removed'] == 0])}")
print(f"  From final pruned results: {len([td for td in target_dnas if td['weights_removed'] > 0])}")

if target_dnas:
    print(f"\n📊 Target DNA Details:")
    for i, dna in enumerate(target_dnas):
        print(f"  {i+1}. Score: {dna['pruned_score']} | Weights: {dna['pruned_nonzero']} | "
              f"Reduction: {dna['weights_removed']/dna['original_nonzero']*100:.1f}% | "
              f"Source: {'Successful Vector' if dna['weights_removed'] == 0 else 'Final Pruned'}")
else:
    print(f"❌ No DNAs meet both score (>={PRUNING_THRESHOLD}) and weight (<={MAX_PRUNED_WEIGHTS}) criteria")

# Alternative: Use slower but more thorough generation-based method
# pruned_results, successful_vectors = prune_dna_vectors(high_scoring_dnas, PRUNING_THRESHOLD, method="generation_based")


🎯 TARGET DNAs FOUND: 1 meet both criteria:
  Score >= 975: ✅
  Weights <= 18: ✅
  From successful vectors: 1
  From final pruned results: 0

📊 Target DNA Details:
  1. Score: 980 | Weights: 18 | Reduction: 0.0% | Source: Successful Vector


## Step 3: Create Interactive Visualization Functions

In [15]:
# Visualization functions are now imported from dna_visualization.py module
print("📁 Visualization functions loaded from dna_visualization.py")

📁 Visualization functions loaded from dna_visualization.py


In [16]:
# Generate simulation results using functions from dna_simulation module
try:
    if target_dnas:
        print("\n🧮 Step 4: Generating simulation results for TARGET DNAs...")
        dnas_to_simulate = target_dnas
        simulation_results = generate_all_simulation_results(target_dnas)
    elif pruned_results:
        print("\n🧮 Step 4: Generating simulation results for all pruned DNAs...")
        dnas_to_simulate = pruned_results
        simulation_results = generate_all_simulation_results(pruned_results)
    else:
        dnas_to_simulate = []
        simulation_results = []
        print("❌ No pruned results available for simulation")
except NameError:
    print("⚠️ pruned_results not defined - run the pruning step first")
    dnas_to_simulate = []
    simulation_results = []


🧮 Step 4: Generating simulation results for TARGET DNAs...
🧮 Generating simulation results for 1 DNAs...
  Simulating DNA 1/1...   Running experimental condition...
    ✅ experimental: score=481, missed=18 points
  Running control condition...
    ✅ control: score=499, missed=1 points
✅

✅ Simulation complete: 1 successful simulations


## Step 5: Interactive DNA Browser with Slider

In [17]:
# Create and display the dual DNA browser using functions from dna_browser module
if dnas_to_simulate and simulation_results:
    viz_type = "TARGET" if target_dnas else "ALL PRUNED"
    print(f"\n🎛️ Step 5: Creating interactive dual DNA browser for {viz_type} DNAs...")
    print(f"📊 Browser ready with {len(dnas_to_simulate)} DNA vectors")
    
    if target_dnas:
        print(f"🎯 Showing TARGET DNAs that meet criteria:")
        print(f"  Score >= {PRUNING_THRESHOLD} AND Weights <= {MAX_PRUNED_WEIGHTS}")
    
    print("\nControls:")
    print("• Sort dropdown: Change ordering of DNAs")
    print("• Show dropdown: Choose between voltage traces, network graph, or both")
    print("• DNA slider: Browse through different DNA solutions")
    print("\nEach view shows:")
    print("• Voltage traces: Experimental vs control conditions with stimulus markers")
    print("• Network graph: Directed connectivity with edge weights and inhibitory markers")
    print("• Gold borders highlight neurons used for fitness evaluation")
    print("• 🎯 TARGET marker shows DNAs meeting both score and weight criteria\n")
    
    browser = create_dual_dna_browser(target_dnas, pruned_results, simulation_results, 
                                     PRUNING_THRESHOLD, MAX_PRUNED_WEIGHTS)
    display(browser)
else:
    print("❌ Cannot create browser: No data available")
    print("\nCheck:")
    print("1. Results folder path is correct")
    print("2. Score threshold is appropriate")
    print("3. Aggregated results files exist in subfolders")


🎛️ Step 5: Creating interactive dual DNA browser for TARGET DNAs...
📊 Browser ready with 1 DNA vectors
🎯 Showing TARGET DNAs that meet criteria:
  Score >= 975 AND Weights <= 18

Controls:
• Sort dropdown: Change ordering of DNAs
• Show dropdown: Choose between voltage traces, network graph, or both
• DNA slider: Browse through different DNA solutions

Each view shows:
• Voltage traces: Experimental vs control conditions with stimulus markers
• Network graph: Directed connectivity with edge weights and inhibitory markers
• Gold borders highlight neurons used for fitness evaluation
• 🎯 TARGET marker shows DNAs meeting both score and weight criteria



## Data Export & Summary

In [18]:
# Save all results for later use, including target DNAs
if pruned_results:
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    export_file = f"multiple_ga_analysis_{timestamp}.pkl"
    
    export_data = {
        'config': {
            'results_folder': RESULTS_FOLDER,
            'score_threshold': SCORE_THRESHOLD,
            'pruning_threshold': PRUNING_THRESHOLD,
            'max_dnas_processed': MAX_DNAS_TO_PROCESS,
            'max_pruned_weights': MAX_PRUNED_WEIGHTS
        },
        'high_scoring_dnas': high_scoring_dnas,
        'pruned_results': pruned_results,
        'target_dnas': target_dnas,  # Add target DNAs to export
        'simulation_results': simulation_results,
        'timestamp': timestamp
    }
    
    with open(export_file, 'wb') as f:
        pickle.dump(export_data, f)
    
    print(f"💾 All results exported to: {export_file}")
    
    # Create summary DataFrame for all pruned results
    summary_data = []
    for i, result in enumerate(pruned_results):
        is_target = target_dnas and result in target_dnas
        
        # Handle different original_dna formats
        orig_dna = result['original_dna']
        if isinstance(orig_dna, dict):
            run_folder = orig_dna.get('run_folder', 'Unknown')
            generation = orig_dna.get('generation', 'Unknown')
            process_id = orig_dna.get('process_id', 'Unknown')
        else:
            run_folder = 'Unknown'
            generation = 'Unknown'
            process_id = 'Unknown'
        
        summary_data.append({
            'DNA_ID': i + 1,
            'Is_Target': '🎯' if is_target else '',
            'Original_Score': result['original_score'],
            'Pruned_Score': result['pruned_score'],
            'Exp_Score': result['final_exp_score'],
            'Cont_Score': result['final_cont_score'],
            'Original_Weights': result['original_nonzero'],
            'Pruned_Weights': result['pruned_nonzero'],
            'Weights_Removed': result['weights_removed'],
            'Reduction_Percent': result['weights_removed'] / result['original_nonzero'] * 100,
            'Run_Folder': run_folder,
            'Generation': generation,
            'Process_ID': process_id
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\n📊 FINAL SUMMARY:")
    print("=" * 60)
    print(f"Results folder analyzed: {RESULTS_FOLDER}")
    print(f"High-scoring DNAs found: {len(high_scoring_dnas)} (threshold: {SCORE_THRESHOLD})")
    print(f"Successfully pruned: {len(pruned_results)}")
    print(f"TARGET DNAs found: {len(target_dnas)} (score >= {PRUNING_THRESHOLD}, weights <= {MAX_PRUNED_WEIGHTS})")
    print(f"Average weight reduction: {summary_df['Reduction_Percent'].mean():.1f}%")
    print(f"Best pruned score: {summary_df['Pruned_Score'].max()}")
    print(f"Most efficient (fewest weights): {summary_df['Pruned_Weights'].min()} weights")
    print(f"Total original weights: {summary_df['Original_Weights'].sum()}")
    print(f"Total pruned weights: {summary_df['Pruned_Weights'].sum()}")
    
    # Save summary CSV
    csv_file = f"multiple_ga_summary_{timestamp}.csv"
    summary_df.to_csv(csv_file, index=False)
    print(f"\n📄 Summary table saved to: {csv_file}")
    
    # Display top results
    print("\n🏆 Top 10 Results by Pruned Score:")
    display(summary_df.nlargest(10, 'Pruned_Score')[['DNA_ID', 'Is_Target', 'Pruned_Score', 'Pruned_Weights', 'Reduction_Percent', 'Run_Folder']])
    
    print("\n🎯 Most Efficient (Fewest Final Weights):")
    display(summary_df.nsmallest(10, 'Pruned_Weights')[['DNA_ID', 'Is_Target', 'Pruned_Score', 'Pruned_Weights', 'Reduction_Percent', 'Run_Folder']])
    
    # Show target DNA summary if any found
    if target_dnas:
        # Create summary for target DNAs with safe field access
        target_summary = []
        for i, target in enumerate(target_dnas):
            orig_dna = target['original_dna']
            if isinstance(orig_dna, dict):
                run_folder = orig_dna.get('run_folder', 'Successful Vector')
                generation = orig_dna.get('generation', orig_dna.get('pass', 'Unknown'))
            else:
                run_folder = 'Unknown'
                generation = 'Unknown'
                
            target_summary.append({
                'Target_ID': i + 1,
                'Pruned_Score': target['pruned_score'],
                'Pruned_Weights': target['pruned_nonzero'],
                'Reduction_Percent': target['weights_removed'] / target['original_nonzero'] * 100 if target['original_nonzero'] > 0 else 0,
                'Run_Folder': run_folder,
                'Generation': generation
            })
        
        target_df = pd.DataFrame(target_summary)
        print(f"\n🎯 TARGET DNAs ({len(target_dnas)} found):")
        display(target_df)
        
        # Save target DNAs separately
        target_file = f"target_dnas_{timestamp}.pkl"
        with open(target_file, 'wb') as f:
            pickle.dump({
                'target_dnas': target_dnas,
                'criteria': {
                    'min_score': PRUNING_THRESHOLD,
                    'max_weights': MAX_PRUNED_WEIGHTS
                },
                'timestamp': timestamp
            }, f)
        print(f"🎯 Target DNAs saved separately to: {target_file}")
    
else:
    print("❌ No results to export")

💾 All results exported to: multiple_ga_analysis_20250905_014059.pkl

📊 FINAL SUMMARY:
Results folder analyzed: results/continuous_runs_H_opt4_thresh970_20250902_173402/successful_run_018
High-scoring DNAs found: 3 (threshold: 970)
Successfully pruned: 3
TARGET DNAs found: 1 (score >= 975, weights <= 18)
Average weight reduction: 72.8%
Best pruned score: 970
Most efficient (fewest weights): 13 weights
Total original weights: 147
Total pruned weights: 40

📄 Summary table saved to: multiple_ga_summary_20250905_014059.csv

🏆 Top 10 Results by Pruned Score:


,DNA_ID,Is_Target,Pruned_Score,Pruned_Weights,Reduction_Percent,Run_Folder
1,2,,970,14,71.428571,successful_run_018
0,1,,969,13,72.916667,successful_run_018
2,3,,969,13,74.000000,successful_run_018



🎯 Most Efficient (Fewest Final Weights):


,DNA_ID,Is_Target,Pruned_Score,Pruned_Weights,Reduction_Percent,Run_Folder
0,1,,969,13,72.916667,successful_run_018
2,3,,969,13,74.000000,successful_run_018
1,2,,970,14,71.428571,successful_run_018



🎯 TARGET DNAs (1 found):


,Target_ID,Pruned_Score,Pruned_Weights,Reduction_Percent,Run_Folder,Generation
0,1,980,18,0.0,Successful Vector,2


🎯 Target DNAs saved separately to: target_dnas_20250905_014059.pkl
